In [2]:
!python3.13 -m pip install ipykernel pandas requests

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("/Users/adrienkamdem/Millennium/ai-theme-association/data/SEC_Filing/sec_filings_2023_present.csv")

In [6]:
df

,ticker,company,cik,form,filingDate,reportDate,accessionNumber,primaryDocument,doc_url
0,ADBE,ADOBE INC.,796343,8-K,2026-09-10,2026-09-10,0000796343-26-000147,adbe-20260910.htm,https://www.sec.gov/Archives/edgar/data/796343...
1,ADBE,ADOBE INC.,796343,8-K,2026-09-08,2026-09-02,0000796343-26-000144,adbe-20260902.htm,https://www.sec.gov/Archives/edgar/data/796343...
2,ADBE,ADOBE INC.,796343,8-K,2026-07-17,2026-07-14,0000796343-26-000120,adbe-20260714.htm,https://www.sec.gov/Archives/edgar/data/796343...
3,ADBE,ADOBE INC.,796343,10-Q,2026-06-15,2026-05-29,0000796343-26-000112,adbe-20260529.htm,https://www.sec.gov/Archives/edgar/data/796343...
4,ADBE,ADOBE INC.,796343,8-K,2026-06-11,2026-06-08,0000796343-26-000109,adbe-20260608.htm,https://www.sec.gov/Archives/edgar/data/796343...
...,...,...,...,...,...,...,...,...,...
557,WM,WASTE MANAGEMENT INC,823768,8-K,2023-03-10,2023-03-07,0001104659-23-030666,tm238966d1_8k.htm,https://www.sec.gov/Archives/edgar/data/823768...
558,WM,WASTE MANAGEMENT INC,823768,8-K,2023-02-13,2023-02-08,0001104659-23-019722,tm235749d5_8k.htm,https://www.sec.gov/Archives/edgar/data/823768...
559,WM,WASTE MANAGEMENT INC,823768,10-K,2023-02-07,2022-12-31,0001558370-23-000964,wm-20221231x10k.htm,https://www.sec.gov/Archives/edgar/data/823768...
560,WM,WASTE MANAGEMENT INC,823768,8-K,2023-02-06,2023-02-06,0001104659-23-011281,tm235358d1_8k.htm,https://www.sec.gov/Archives/edgar/data/823768...


In [11]:
df["company"].value_counts()

company
PROCTER & GAMBLE Co       83
Dell Technologies Inc.    76
INTEL CORP                75
WASTE MANAGEMENT INC      66
MICRON TECHNOLOGY INC     60
NVIDIA CORP               53
CATERPILLAR INC           52
Atlassian Corp            50
ADOBE INC.                47
Name: count, dtype: int64

In [13]:
# Make sure filingDate is datetime
df["filingDate"] = pd.to_datetime(df["filingDate"])

# Extract year and quarter
df["year"] = df["filingDate"].dt.year
df["quarter"] = "Q" + df["filingDate"].dt.quarter.astype(str)

# Count documents per company per quarter
quarterly_counts = (
    df.groupby(["ticker", "year", "quarter"])
      .size()
      .reset_index(name="total_docs")
      .sort_values(["ticker", "year", "quarter"])
)

quarterly_counts

,ticker,year,quarter,total_docs
0,ADBE,2023,Q1,5
1,ADBE,2023,Q2,3
2,ADBE,2023,Q3,2
3,ADBE,2023,Q4,3
4,ADBE,2024,Q1,4
...,...,...,...,...
130,WM,2025,Q3,4
131,WM,2025,Q4,3
132,WM,2026,Q1,5
133,WM,2026,Q2,4


In [14]:
quarterly_by_type = (
    df.groupby(["ticker", "year", "quarter", "form"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

form_columns = [
    col for col in quarterly_by_type.columns
    if col not in ["ticker", "year", "quarter"]
]

quarterly_by_type["total_docs"] = (
    quarterly_by_type[form_columns].sum(axis=1)
)

quarterly_by_type

form,ticker,year,quarter,10-K,10-Q,8-K,total_docs
0,ADBE,2023,Q1,1,1,3,5
1,ADBE,2023,Q2,0,1,2,3
2,ADBE,2023,Q3,0,1,1,2
3,ADBE,2023,Q4,0,0,3,3
4,ADBE,2024,Q1,1,1,2,4
...,...,...,...,...,...,...,...
130,WM,2025,Q3,0,1,3,4
131,WM,2025,Q4,0,1,2,3
132,WM,2026,Q1,1,0,4,5
133,WM,2026,Q2,0,1,3,4


In [15]:
company_stats = (
    quarterly_counts
    .groupby("ticker")["total_docs"]
    .agg(
        avg_docs_per_quarter="mean",
        median_docs_per_quarter="median",
        min_docs_per_quarter="min",
        max_docs_per_quarter="max",
        std_docs_per_quarter="std"
    )
    .round(2)
    .reset_index()
)

company_stats

,ticker,avg_docs_per_quarter,median_docs_per_quarter,min_docs_per_quarter,max_docs_per_quarter,std_docs_per_quarter
0,ADBE,3.13,3.0,1,5,1.25
1,CAT,3.47,3.0,2,6,1.13
2,DELL,5.07,5.0,2,8,1.71
3,INTC,5.00,5.0,3,11,2.20
4,MU,4.00,4.0,1,6,1.36
5,NVDA,3.53,3.0,2,6,0.99
6,PG,5.53,6.0,4,8,1.41
7,TEAM,3.33,3.0,2,6,1.23
8,WM,4.40,4.0,3,7,1.24


In [16]:
complete_quarters = quarterly_counts[
    ~(
        (quarterly_counts["year"] == 2026)
        & (quarterly_counts["quarter"] == "Q3")
    )
]

company_stats = (
    complete_quarters
    .groupby("ticker")["total_docs"]
    .agg(
        avg_docs_per_quarter="mean",
        median_docs_per_quarter="median",
        min_docs_per_quarter="min",
        max_docs_per_quarter="max",
        std_docs_per_quarter="std"
    )
    .round(2)
    .reset_index()
)

In [17]:
company_stats

,ticker,avg_docs_per_quarter,median_docs_per_quarter,min_docs_per_quarter,max_docs_per_quarter,std_docs_per_quarter
0,ADBE,3.14,3.0,1,5,1.29
1,CAT,3.50,3.5,2,6,1.16
2,DELL,5.07,5.0,2,8,1.77
3,INTC,5.14,5.0,3,11,2.21
4,MU,4.21,4.0,2,6,1.12
5,NVDA,3.43,3.0,2,6,0.94
6,PG,5.57,6.0,4,8,1.45
7,TEAM,3.43,3.5,2,6,1.22
8,WM,4.50,4.0,3,7,1.22
